## Setup

In [1]:
from google.colab import userdata
access_token = userdata.get('CASM-NER')

In [3]:
%%capture
!pip install transformers
!pip install sentencepiece
!pip install seqeval
!pip install datasets
# !pip install git+https://github.com/ay94/multilingual-ner.git

In [ ]:
# from ner import evaluation

In [4]:
## Mount GDrive
from google.colab import drive
drive.mount('/content/drive/', force_remount=True)

## Imports
import os
import sys
import nltk
import time
import torch
import random
import subprocess
import numpy as np
import pandas as pd
import datetime as dt
from itertools import groupby
from tqdm.notebook import tqdm
from datasets import load_dataset
from transformers import pipeline
from collections import Counter, defaultdict
from torch.utils.data import DataLoader, Dataset
from transformers import AutoModelForTokenClassification, AutoTokenizer
from seqeval.metrics import f1_score as seq_f1, precision_score as seq_precision, recall_score as seq_recall, classification_report as seq_classification
from sklearn.metrics import f1_score as skl_f1, precision_score as skl_precision, recall_score as skl_recall, classification_report as skl_classification

Mounted at /content/drive/


In [5]:
# Append the library files into the notebook system path for import
sys.path.append('/content/drive/Shareddrives/Machine Translation/Model benchmarking/Libraries/1.0.2')
# import custom library files
import ner, utils

## Load datasets

### wikiann

In [ ]:
# from ner.dataset_base import HuggingFaceMultilingualDataset
# class Wikiann(HuggingFaceMultilingualDataset):
#     dataset_name = 'wikiann'
#     language = 'ro'
#     license = 'unknown'

# dataset = Wikiann()
# dataset.check_labels()

In [6]:
wikiann_label_map = {
    "O": 0,
    "B-PER": 1,
    "I-PER": 2,
    "B-ORG": 3,
    "I-ORG": 4,
    "B-LOC": 5,
    "I-LOC": 6
}

wikiann = ner.ReadNERData()
wikiann_words, wikiann_labels = wikiann.read_dataset('wikiann', wikiann_label_map, lang='ro')

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:72: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Generating validation split:   0%|          | 0/10000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/10000 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/20000 [00:00<?, ? examples/s]

Generating test Split


  0%|          | 0/10000 [00:00<?, ?it/s]

In [7]:
print(ner.check_labels(wikiann_labels))
# Dataset Label Map Alignment to LOC, ORG, PERS, MISC

{'O', 'I-PER', 'B-ORG', 'B-LOC', 'I-LOC', 'I-ORG', 'B-PER'}


### Ronec
https://huggingface.co/datasets/ronec


In [9]:
ronec_label_map = {
    'O': 0,
    'B-PERSON': 1,
    'I-PERSON': 2,
    'B-GPE': 3,
    'I-GPE': 4,
    'B-LOC': 5,
    'I-LOC': 6,
    'B-ORG': 7,
    'I-ORG': 8,
    'B-LANGUAGE': 9,
    'I-LANGUAGE': 10,
    'B-NAT_REL_POL': 11 ,
    'I-NAT_REL_POL': 12,
    'B-DATETIME': 13,
    'I-DATETIME': 14,
    'B-PERIOD': 15,
    'I-PERIOD': 16,
    'B-QUANTITY': 17,
    'I-QUANTITY': 18,
    'B-MONEY': 19,
    'I-MONEY': 20,
    'B-NUMERIC': 21,
    'I-NUMERIC': 22,
    'B-ORDINAL': 23,
    'I-ORDINAL': 24,
    'B-FACILITY': 25,
    'I-FACILITY': 26,
    'B-WORK_OF_ART': 27 ,
    'I-WORK_OF_ART': 28,
    'B-EVENT': 29,
    'I-EVENT': 30,
}

ronec = ner.ReadNERData()
ronec_words, ronec_labels = ronec.read_dataset('ronec', ronec_label_map)

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:72: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Generating test Split


  0%|          | 0/2000 [00:00<?, ?it/s]

In [11]:
print(ner.check_labels(ronec_labels))
label_alignment = {
    'O': 'O',
    'B-PERSON': 'B-PER',
    'I-PERSON': 'I-PER',
    'B-GPE': 'O',
    'I-GPE': 'O',
    'B-LOC': 'B-LOC',
    'I-LOC': 'I-LOC',
    'B-ORG': 'B-ORG',
    'I-ORG': 'I-ORG',
    'B-LANGUAGE': 'O',
    'I-LANGUAGE': 'O',
    'B-NAT_REL_POL': 'O',
    'I-NAT_REL_POL': 'O',
    'B-DATETIME': 'O',
    'I-DATETIME': 'O',
    'B-PERIOD': 'O',
    'I-PERIOD': 'O',
    'B-QUANTITY': 'O',
    'I-QUANTITY': 'O',
    'B-MONEY': 'O',
    'I-MONEY': 'O',
    'B-NUMERIC': 'O',
    'I-NUMERIC': 'O',
    'B-ORDINAL': 'O',
    'I-ORDINAL': 'O',
    'B-FACILITY': 'O',
    'I-FACILITY': 'O',
    'B-WORK_OF_ART': 'O',
    'I-WORK_OF_ART': 'O',
    'B-EVENT': 'O',
    'I-EVENT': 'O',
}
# Align the dataset labels to the standard labels
ronec_labels = ner.align_dataset(ronec_labels, label_alignment)
print(ner.check_labels(ronec_labels))

{'B-QUANTITY', 'I-GPE', 'B-DATETIME', 'I-NAT_REL_POL', 'B-FACILITY', 'B-ORDINAL', 'I-ORG', 'I-NUMERIC', 'I-ORDINAL', 'I-WORK_OF_ART', 'I-MONEY', 'B-LOC', 'I-DATETIME', 'O', 'B-PERIOD', 'B-GPE', 'B-WORK_OF_ART', 'I-PERIOD', 'B-NAT_REL_POL', 'I-QUANTITY', 'B-LANGUAGE', 'B-EVENT', 'B-PERSON', 'I-EVENT', 'I-FACILITY', 'B-MONEY', 'I-LANGUAGE', 'B-NUMERIC', 'B-ORG', 'I-LOC', 'I-PERSON'}
{'O', 'I-PER', 'B-ORG', 'B-LOC', 'I-LOC', 'I-ORG', 'B-PER'}


# Evaluate model

In [12]:
alignment = {
'B-organization': 'B-ORG',
'O': 'O',
'B-other': 'O',
'B-person': 'B-PER',
'I-person': 'I-PER',
'B-location': 'B-LOC',
'I-organization': 'I-ORG',
'I-other': 'O',
'I-location': 'I-LOC'
}

model_name = "tner/xlm-roberta-large-conll2003"
model_name_output = 'tner-xlm-roberta-large'
model_evaluation = ner.ModelEvaluation(
    model_name,
    alignment
)

tokenizer_config.json:   0%|          | 0.00/212 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.01k [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/2.24G [00:00<?, ?B/s]

In [ ]:
# model_evaluation.model.config.id2label

{0: 'B-LOC',
 1: 'B-MISC',
 2: 'B-ORG',
 3: 'I-LOC',
 4: 'I-MISC',
 5: 'I-ORG',
 6: 'I-PER',
 7: 'O'}

### wikiann

In [13]:
data_name = "wikiann"
wikiann_evaluation_output = model_evaluation.evaluate_model(wikiann_words, wikiann_labels)

  0%|          | 0/625 [00:00<?, ?it/s]

In [14]:
wikiann_seqeval = wikiann_evaluation_output.get_classification('Seqeval')
wikiann_seqeval

,Tag,Precision,Recall,F1,support
0,LOC,0.1899,0.3202,0.2384,3804
1,ORG,0.5786,0.3895,0.4656,3666
2,PER,0.6745,0.7235,0.6981,4220
3,micro,0.4250,0.4875,0.4541,11690
4,macro,0.4810,0.4777,0.4674,11690
5,weighted,0.4868,0.4875,0.4756,11690


In [15]:
wikiann_sklearn = wikiann_evaluation_output.get_classification('Sklearn')
wikiann_sklearn

,Tag,Precision,Recall,F1,support
0,B-LOC,0.3575,0.5917,0.4457,3804
1,B-ORG,0.7665,0.4853,0.5943,3666
2,B-PER,0.8385,0.8794,0.8584,4220
3,I-LOC,0.6266,0.1848,0.2854,7583
4,I-ORG,0.8973,0.3701,0.5240,10104
5,I-PER,0.9630,0.5914,0.7328,8314
6,O,0.6788,0.9864,0.8042,28991
7,accuracy,0.6958,66682,None,None
8,macro,0.7326,0.5842,0.6064,66682
9,weighted,0.7380,0.6958,0.6653,66682


### Ronec

In [16]:
data_name = "ronec"
ronec_evaluation_output = model_evaluation.evaluate_model(ronec_words, ronec_labels)

  0%|          | 0/125 [00:00<?, ?it/s]

In [17]:
ronec_seqeval = ronec_evaluation_output.get_classification('Seqeval')
ronec_seqeval

,Tag,Precision,Recall,F1,support
0,LOC,0.6876,0.9450,0.7960,1728
1,ORG,0.0006,0.0027,0.0010,373
2,PER,0.6818,0.2908,0.4077,4230
3,micro,0.4867,0.4524,0.4689,6331
4,macro,0.4567,0.4128,0.4016,6331
5,weighted,0.6433,0.4524,0.4897,6331


In [18]:
ronec_sklearn = ronec_evaluation_output.get_classification('Sklearn')
ronec_sklearn

,Tag,Precision,Recall,F1,support
0,B-LOC,0.7166,0.9554,0.8189,1728
1,B-ORG,0.0013,0.0054,0.0020,373
2,B-PER,0.7503,0.3161,0.4448,4230
3,I-LOC,0.3781,0.8462,0.5226,273
4,I-ORG,0.0042,0.0179,0.0068,503
5,I-PER,0.9672,0.5526,0.7033,2186
6,O,0.9517,0.9470,0.9494,79176
7,accuracy,0.8977,88469,None,None
8,macro,0.5385,0.5201,0.4926,88469
9,weighted,0.9267,0.8977,0.9060,88469


In [19]:
alignment = {
'B-LOC': 'B-LOC',
'B-MISC': 'O',
'B-ORG': 'B-ORG',
'I-LOC': 'I-LOC',
'I-MISC': 'O',
'I-ORG': 'I-ORG',
'I-PER': 'I-PER',
'O': 'O'
}

model_name = "xlm-roberta-large-finetuned-conll03-english"
model_name_output = 'xlm-roberta-large'
model_evaluation = ner.ModelEvaluation(
    model_name,
    alignment
)

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:72: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/852 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.24G [00:00<?, ?B/s]

Some weights of the model checkpoint at xlm-roberta-large-finetuned-conll03-english were not used when initializing XLMRobertaForTokenClassification: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing XLMRobertaForTokenClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing XLMRobertaForTokenClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


### wikiann

In [20]:
data_name = "wikiann"
wikiann_evaluation_output = model_evaluation.evaluate_model(wikiann_words, wikiann_labels)

  0%|          | 0/625 [00:00<?, ?it/s]

/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classif

In [21]:
wikiann_seqeval = wikiann_evaluation_output.get_classification('Seqeval')
wikiann_seqeval

,Tag,Precision,Recall,F1,support
0,LOC,0.1981,0.3320,0.2481,3804
1,ORG,0.5800,0.3816,0.4603,3666
2,PER,0.6385,0.7066,0.6709,4220
3,micro,0.4194,0.4828,0.4489,11690
4,macro,0.4722,0.4734,0.4598,11690
5,weighted,0.4769,0.4828,0.4673,11690


In [22]:
wikiann_sklearn = wikiann_evaluation_output.get_classification('Sklearn')
wikiann_sklearn

,Tag,Precision,Recall,F1,support
0,B-LOC,0.0000,0.0000,0.0000,3804
1,B-ORG,0.0000,0.0000,0.0000,3666
2,B-PER,0.0000,0.0000,0.0000,4220
3,I-LOC,0.4127,0.4849,0.4459,7583
4,I-ORG,0.5859,0.3664,0.4509,10104
5,I-PER,0.5257,0.5966,0.5589,8314
6,O,0.6804,0.9861,0.8052,28991
7,accuracy,0.6138,66682,None,None
8,macro,0.3150,0.3477,0.3230,66682
9,weighted,0.4971,0.6138,0.5388,66682


### Ronec

In [ ]:
data_name = "ronec"
ronec_evaluation_output = model_evaluation.evaluate_model(ronec_words, ronec_labels)

  0%|          | 0/125 [00:00<?, ?it/s]

/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classif

In [ ]:
ronec_seqeval = ronec_evaluation_output.get_classification('Seqeval')
ronec_seqeval

,Tag,Precision,Recall,F1,support
0,LOC,0.6741,0.9398,0.7851,1728
1,ORG,0.0000,0.0000,0.0000,373
2,PER,0.6935,0.2948,0.4137,4230
3,micro,0.4872,0.4535,0.4697,6331
4,macro,0.4559,0.4115,0.3996,6331
5,weighted,0.6474,0.4535,0.4907,6331


In [ ]:
ronec_sklearn = ronec_evaluation_output.get_classification('Sklearn')
ronec_sklearn

,Tag,Precision,Recall,F1,support
0,B-LOC,0.0000,0.0000,0.0000,1728
1,B-ORG,0.0000,0.0000,0.0000,373
2,B-PER,0.0000,0.0000,0.0000,4230
3,I-LOC,0.0808,0.9084,0.1484,273
4,I-ORG,0.0013,0.0099,0.0024,503
5,I-PER,0.5246,0.7315,0.6110,2186
6,O,0.9524,0.9457,0.9490,79176
7,accuracy,0.8673,88469,None,None
8,macro,0.2227,0.3708,0.2444,88469
9,weighted,0.8656,0.8673,0.8649,88469
